In [2]:
import pandas as pd
import trafilatura
# import requests
from curl_cffi import requests
from tqdm import tqdm, trange
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import re
import sys
from typing import Optional

In [3]:
# Support Functions

def scrape_for_llm_DEPRECATED(url: str) -> str:
    """
    Downloads webpage content and strips non-essential boilerplate,
    returning clean Markdown suitable for LLM context windows.
    """
    
    extracted_text, success, error_type = '', True, ''
    html_content = trafilatura.fetch_url(url)
    if not html_content:
        success = False
        error_type = 'Could not fetch HTML from URL'
        return extracted_text, success, error_type
    
    extracted_text = trafilatura.extract(
        html_content,
        output_format="markdown",
        include_links=True,
        include_images=False
    )
    
    if not extracted_text:
        success = False
        error_type = 'Could not extract readable text from URL content'
        
    return extracted_text, success, error_type

def scrape_for_llm(url: str, timeout: tuple[float, float] = (10.0, 10.0)) -> str:
    """
    Downloads webpage content with a strict timeout and extracts clean markdown.
    
    :param url: Target LP URL
    :param timeout: Tuple of (connect_timeout, read_timeout) in seconds
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    extracted_text, success, error_type = '', True, ''

    # V2
    # try:
    #     # Pass (connection limit, response read limit)
    #     response = requests.get(url, headers=headers, timeout=timeout)
    #     response.raise_for_status()
    #     html_content = response.text
    # except requests.exceptions.ConnectTimeout:
    #     success, error_type = False, 'Connection timed out while reaching the LP'
    # except requests.exceptions.ReadTimeout:
    #     success, error_type = False, 'The LP connected, but timed out sending data'
    # except requests.exceptions.RequestException as e:
    #     success, error_type = False, 'Failed to fetch content from URL: '+str(e)

    # # V3
    # try:
    #     # impersonate="chrome" sends Chrome's exact TLS fingerprint and headers
    #     response = requests.get(url, impersonate="chrome", timeout=timeout[1])
    #     response.raise_for_status()
    #     html_content = response.text
    # except Exception as e:
    #     success, error_type = False, 'Failed to fetch content from URL: '+str(e)

    # V4
    try:
        HEADERS = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Accept-Encoding": "gzip, deflate, br",
            "Sec-Ch-Ua": '"Google Chrome";v="125", "Chromium";v="125", "Not.A/Brand";v="24"',
            "Sec-Ch-Ua-Mobile": "?0",
            "Sec-Ch-Ua-Platform": '"Windows"',
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Upgrade-Insecure-Requests": "1"
        }
        # Pass (connection limit, response read limit)
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        html_content = response.text
    except requests.exceptions.ConnectTimeout:
        success, error_type = False, 'Connection timed out while reaching the LP'
    except requests.exceptions.ReadTimeout:
        success, error_type = False, 'The LP connected, but timed out sending data'
    except requests.exceptions.RequestException as e:
        success, error_type = False, 'Failed to fetch content from URL: '+str(e)

    if not success:
        return extracted_text, success, error_type

    extracted_text = trafilatura.extract(
        html_content,
        output_format="markdown",
        include_links=True,
        include_images=False
    )

    if not extracted_text:
        success, error_type = False, 'Could not extract readable text from URL content'

    return extracted_text, success, error_type

def get_rendered_html(url):
    with sync_playwright() as p:
        # Launch a headless browser
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        
        # Navigate to the URL and wait until the network is idle
        page.goto(url, wait_until="networkidle")
        
        # Extract the fully rendered HTML
        html = page.content()
        
        browser.close()
        return html

def html_to_llm_text(html: str, url: Optional[str] = None, include_metadata: bool = False) -> str:
    """
    Convert rendered HTML into clean, readable plain text for an LLM prompt.

    Strips nav bars, ads, scripts, and other boilerplate; keeps the main
    body content and (optionally) prepends title/author/date metadata.

    Args:
        html: Raw HTML string (already rendered — e.g. page.content()
              from a headless browser, or HTML you pasted/saved yourself).
        url: Optional source URL. Not fetched — only used to help
             trafilatura resolve metadata/relative links more accurately.
        include_metadata: If True, prepends a small metadata block
              (title/author/date/source) above the extracted text.

    Returns:
        Clean text string, ready to paste into a prompt.
    """

    # --- Fallback path: manual cleanup if trafilatura can't extract confidently ---
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form", "noscript", "svg"]):
        tag.decompose()

    for tag in soup.find_all(attrs={"role": ["navigation", "banner", "contentinfo"]}):
        tag.decompose()

    title_tag = soup.find("title")
    title = title_tag.get_text(strip=True) if title_tag else None
    if include_metadata and title:
        metadata_block = f"Title: {title}\n\n"

    text = soup.get_text(separator="\n")
    text = re.sub(r"[ \t]+", " ", text)          # collapse repeated spaces/tabs
    text = re.sub(r"\n\s*\n+", "\n\n", text)      # collapse repeated blank lines
    text = text.strip()

    return metadata_block + text

In [ ]:

df = pd.read_excel('AQR eval content-type-agreement (2 or 3 Welo ER).xlsx',sheet_name='Sheet1')
df = df[df['Ad Type'].isin(['shopping','text'])]
fname = 'AQR_data_with_LP_content.xlsx'

print('>> Scrapping landing page content')
data = {'LP content':[],'LP scrapping succeeded':[],'LP scrapping error status':[]}
for i in tqdm(range(len(df))):
    lp_url = str(df['land_page'].iloc[i])
    
    lp_content, success, error = scrape_for_llm(lp_url)
    data['LP content'].append(lp_content)
    data['LP scrapping succeeded'].append(str(success))
    data['LP scrapping error status'].append(error)
print('>> Done')

ext_df = pd.concat([df, pd.DataFrame(data)], axis=1)
ext_df.to_excel(fname,index=False)
print('>> Saved landing page content in '+fname)

>> Scrapping landing page content


 37%|███▋      | 1715/4593 [37:20<1:10:04,  1.46s/it]